In [ ]:
import os
import glob
import logging
import pandas as pd
from io import BytesIO
from PIL import Image
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from msrest.authentication import CognitiveServicesCredentials
from azure.storage.blob import BlobServiceClient, ContentSettings
from sqlalchemy import create_engine
import urllib.parse

# ——————————————————————————————
#  AZURE COMPUTER VISION (OCR) CONFIGURATION
# ——————————————————————————————
# Endpoint URL and subscription key for your Azure Computer Vision resource
OCR_ENDPOINT = "https://chatter-ocr.cognitiveservices.azure.com/"
OCR_KEY      = "D50WCTFHBRNwR1I301mnTIUn4lpjyHDtRO8zHjlBpCJq7kfh7ci2JQQJ99BEACYeBjFXJ3w3AAAFACOG7Lgv"

# Create a client object for the Computer Vision service
cv_client = ComputerVisionClient(
    OCR_ENDPOINT,
    CognitiveServicesCredentials(OCR_KEY)
)


# ——————————————————————————————
#  AZURE BLOB STORAGE CONFIGURATION
# ——————————————————————————————
# Connection string and container name for your Azure Blob Storage account
BLOB_CONN_STR = (
    "DefaultEndpointsProtocol=https;"
    "AccountName=chattersa;"
    "AccountKey=7CTXGYtplgRoZWCXjbIbB0RugRVS/XYnrC2BwgZ2BUX+tdaJyj+jWisyG1tYAX75pco2T6c1IFm2+AStzrTdEw==;"
    "EndpointSuffix=core.windows.net"
)
CONTAINER_NAME = "extractedimages"

# Instantiate the BlobServiceClient and get (or create) the container
blob_service     = BlobServiceClient.from_connection_string(BLOB_CONN_STR)
container_client = blob_service.get_container_client(CONTAINER_NAME)
if not container_client.exists():
    container_client.create_container()


# ——————————————————————————————
#  SQL SERVER (ODBC) CONFIGURATION
# ——————————————————————————————
# Build the ODBC connection string and wrap it for SQLAlchemy
odbc_str = (
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=tcp:covid-srv-nicolas.database.windows.net,1433;"
    "Database=chatter;"
    "Uid=adm;"
    "Pwd=Facedes14!;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)
# URL‐encode the ODBC string for SQLAlchemy
conn_str = "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(odbc_str)
engine = create_engine(conn_str)


# ——————————————————————————————
#  LOCAL DIRECTORY WITH YOUR .parquet FILES
# ——————————————————————————————
PARQUET_DIR = r"C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\parquet files"


def process_and_save(parquet_path: str):
    """
    1. Locate all .parquet files under the given directory.
    2. For each file:
       a. Read the DataFrame.
       b. Extract each embedded image, upload it to Blob Storage.
       c. Run OCR on the image bytes.
       d. Collect metadata and OCR text.
    3. Compile everything into a pandas DataFrame.
    4. Write the final DataFrame into a SQL Server table.
    """
    # Find all .parquet files
    parquet_files = glob.glob(os.path.join(parquet_path, "*.parquet"))
    if not parquet_files:
        print("No .parquet files found in", parquet_path)
        return

    # This list will accumulate one dict per image row
    records = []

    for file_path in parquet_files:
        print(f"\n=== Processing file: {os.path.basename(file_path)} ===")
        # Read the parquet file into a pandas DataFrame
        df = pd.read_parquet(file_path, engine="pyarrow")

        # Iterate over each row in that DataFrame
        for idx, row in df.iterrows():
            # --- 1) Extract the raw image bytes and original filename/path ---
            img_struct     = row["image"]
            raw_bytes      = img_struct.get("bytes")
            raw_path_field = img_struct.get("path")
            original_name  = (
                raw_path_field.decode("utf-8", errors="ignore")
                if isinstance(raw_path_field, (bytes, bytearray))
                else raw_path_field
            )

            # --- 2) Use Pillow (PIL) to discover format and dimensions ---
            img = Image.open(BytesIO(raw_bytes))
            image_format = img.format.upper()   # e.g. 'JPEG', 'PNG'
            width, height = img.size            # pixel dimensions

            # --- 3) Upload the image to Azure Blob Storage ---
            #    We strip the extension to build a clean blob name
            image_id  = os.path.splitext(original_name)[0]
            blob_name = f"{image_id}.{image_format.lower()}"
            blob_client = container_client.get_blob_client(blob_name)
            content_settings = ContentSettings(content_type=f"image/{image_format.lower()}")
            # Overwrite=True ensures reruns replace existing blobs
            blob_client.upload_blob(raw_bytes, overwrite=True, content_settings=content_settings)
            blob_url = blob_client.url  # URL to the uploaded image

            # --- 4) Call Azure OCR to extract text from the image ---
            ocr_result = cv_client.recognize_printed_text_in_stream(
                image=BytesIO(raw_bytes),
                language="es"  # specify Spanish OCR
            )

            # --- 5) Build a single string of all words returned by OCR ---
            extracted_text = " ".join(
                word.text
                for region in ocr_result.regions
                for line in region.lines
                for word in line.words
            ).strip() or "<no text>"

            # --- 6) Append a record dict for this image to our list ---
            records.append({
                "Label":       row.get("label", None),
                "Path":        original_name,
                "Format":      image_format,
                "Size":        f"{width} x {height} px",
                "OCR_Text":    extracted_text,
                "BlobURL":     blob_url
            })

    # --- 7) Convert the collected list of dicts into a pandas DataFrame ---
    df_metadata = pd.DataFrame(records)

    # --- 8) Write the DataFrame into SQL Server; creates or replaces the table ---
    df_metadata.to_sql(
        name="ocr_results",  # new table name in your database
        con=engine,
        if_exists="replace",
        index=False
    )
    print(f"Inserted {len(df_metadata)} rows into 'ocr_results' table.")


if __name__ == "__main__":
    # Initialize logging and run the process
    logging.basicConfig(level=logging.INFO)
    process_and_save(PARQUET_DIR)
